In [10]:
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [11]:
def pdf_loader(file_path):
    all_documents = []
    pdf_path = Path(file_path)
    print(pdf_path)
    pdf_files = list(pdf_path.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in {pdf_path}")
    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}...")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"
            

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} documents from {pdf_file}")
        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")
    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents
all_docs = pdf_loader("../data")

../data
Found 3 PDF files in ../data
Processing ../data/pdf/Embeddings.pdf...
Loaded 18 documents from ../data/pdf/Embeddings.pdf
Processing ../data/pdf/Attention.pdf...
Loaded 11 documents from ../data/pdf/Attention.pdf
Processing ../data/pdf/Object_Detection.pdf...
Loaded 22 documents from ../data/pdf/Object_Detection.pdf
Total documents loaded: 51


In [12]:
def chunking(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Total chunks created: {len(split_docs)} from {len(documents)} documents")
    if split_docs:
        print(f"Example chunk content: {split_docs[0].page_content[:200]}...")
        print(f"Example chunk metadata: {split_docs[0].metadata}")
    return split_docs
chunks = chunking(all_docs)

Total chunks created: 289 from 51 documents
Example chunk content: Towards General Text Embeddings with Multi-stage Contrastive Learning
Zehan Li1, Xin Zhang1, Yanzhao Zhang1, Dingkun Long1, Pengjun Xie1, Meishan Zhang
1Alibaba Group
{lizehan.lzh,linzhang.zx,zhangyan...
Example chunk metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-08T01:37:33+00:00', 'source': 'Embeddings.pdf', 'file_path': '../data/pdf/Embeddings.pdf', 'total_pages': 18, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2023-08-08T01:37:33+00:00', 'trapped': '', 'modDate': 'D:20230808013733Z', 'creationDate': 'D:20230808013733Z', 'page': 0, 'file_type': 'pdf'}


In [13]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List ,Tuple ,Any ,Dict
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
class EmbeddingManager:
    def __init__(self,model_name:str="all-MiniLM-l6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()
    def _load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"The model {self.model_name} has dimensions {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(e)
            raise
    def embedding_text(self,text:List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("model not loaded")
        print(f"Genrating embedding for text of length: {len(text)}")
        embedding = self.model.encode(text,show_progress_bar=True)
        print(f"Genrated embedding with shape {embedding.shape}")
        return embedding
    

In [15]:
embedding_model = EmbeddingManager()
embedding_model

The model all-MiniLM-l6-v2 has dimensions 384


In [16]:
class VectorStore:
    def __init__(self, collection_name:str = "pdf_document", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._init_store()

    def _init_store(self):
        try:
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client = chromadb.PersistentClient(self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document embedding for RAG"}
            )
            print(f"Collection:{self.collection_name}")
            print(f"vectors inside collection :{self.collection.count()}")
        except Exception as e:
            print(e)
            raise
    def add_documents(self, document: List[Any], embedding: np.ndarray):
        if len(document) != len(embedding):
            raise ValueError("The number of documents and embeddings must be the same.")
        print(f"Adding {len(document)} documents to the vector store...")

        ids = []
        metadatas = []
        document_texts = []
        embedding_list = []
        for i,(doc,emv) in enumerate(zip(document,embedding)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)
            document_texts.append(doc.page_content)
            embedding_list.append(emv.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embedding_list,
                metadatas=metadatas,
                documents=document_texts
            )
            print(f"Successfully added {len(document)} documents to the vector store.")
            print(f"Total vectors in collection after addition: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to the vector store: {e}")
            raise
vectordb = VectorStore()
vectordb

Collection:pdf_document
vectors inside collection :0


In [17]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-08T01:37:33+00:00', 'source': 'Embeddings.pdf', 'file_path': '../data/pdf/Embeddings.pdf', 'total_pages': 18, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2023-08-08T01:37:33+00:00', 'trapped': '', 'modDate': 'D:20230808013733Z', 'creationDate': 'D:20230808013733Z', 'page': 0, 'file_type': 'pdf'}, page_content='Towards General Text Embeddings with Multi-stage Contrastive Learning\nZehan Li1, Xin Zhang1, Yanzhao Zhang1, Dingkun Long1, Pengjun Xie1, Meishan Zhang\n1Alibaba Group\n{lizehan.lzh,linzhang.zx,zhangyanzhao.zyz,\ndingkun.ldk,pengjun.xpj}@alibaba-inc.com\nAbstract\nWe present GTE, a general-purpose text embed-\nding model trained with multi-stage contrastive\nlearning. In line with recent advancements in\nunifying various NLP tasks into a single for-\nmat, we train a unified text embedding model\nby employing contrastive learn

In [18]:
texts = [chunk.page_content for chunk in chunks]
texts

['Towards General Text Embeddings with Multi-stage Contrastive Learning\nZehan Li1, Xin Zhang1, Yanzhao Zhang1, Dingkun Long1, Pengjun Xie1, Meishan Zhang\n1Alibaba Group\n{lizehan.lzh,linzhang.zx,zhangyanzhao.zyz,\ndingkun.ldk,pengjun.xpj}@alibaba-inc.com\nAbstract\nWe present GTE, a general-purpose text embed-\nding model trained with multi-stage contrastive\nlearning. In line with recent advancements in\nunifying various NLP tasks into a single for-\nmat, we train a unified text embedding model\nby employing contrastive learning over a di-\nverse mixture of datasets from multiple sources.\nBy significantly increasing the number of train-\ning data during both unsupervised pre-training\nand supervised fine-tuning stages, we achieve\nsubstantial performance gains over existing em-\nbedding models. Notably, even with a relatively\nmodest parameter count of 110M, GTEbase out-\nperforms the black-box embedding API pro-\nvided by OpenAI and even surpasses 10x larger\ntext embedding models

In [19]:
embedding = embedding_model.embedding_text(texts)

vectordb.add_documents(chunks,embedding)

Genrating embedding for text of length: 289


Batches: 100%|██████████| 10/10 [00:25<00:00,  2.55s/it]


Genrated embedding with shape (289, 384)
Adding 289 documents to the vector store...
Successfully added 289 documents to the vector store.
Total vectors in collection after addition: 289


In [30]:
class Retriever:
    def __init__(self, vector_store: VectorStore, embedding_model: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_model = embedding_model
    
    def retrive(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:

        print(f"Retrieving documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}")

        query_embedding = self.embedding_model.embedding_text([query])[0]
        
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding],
                n_results=top_k
            )
            print(f"Retrieved {len(results['documents'][0])} results from the vector store.")

            retrived_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                scores = results['distances'][0]
                ids = results['ids'][0]


                for i, (doc_id, doc, metadata, distance) in enumerate(zip(ids, documents, metadatas, scores)):
                    similarity_score = 1.0 / (1.0 + distance)
                    if similarity_score >= score_threshold:
                        retrived_docs.append({
                            "id": doc_id,
                            "document": doc,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })
                        print(f"Result {i+1}: ID={doc_id}, Similarity Score={similarity_score:.4f}, Metadata={metadata}")
                    else:
                        print(f"Result {i+1} below threshold: ID={doc_id}, Similarity Score={similarity_score:.4f}")
                return retrived_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
retriever = Retriever(vectordb,embedding_model)
retriever


In [31]:
retriever.retrive("What is attention is all you need")

Retrieving documents for query: 'What is attention is all you need' with top_k=5 and score_threshold=0.0
Genrating embedding for text of length: 1


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.23it/s]

Genrated embedding with shape (1, 384)
Retrieved 5 results from the vector store.
Result 1: ID=doc_a461efd7_119, Similarity Score=0.4663, Metadata={'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'keywords': '', 'file_type': 'pdf', 'doc_index': 119, 'file_path': '../data/pdf/Attention.pdf', 'content_length': 969, 'producer': 'PyPDF2', 'source': 'Attention.pdf', 'total_pages': 11, 'format': 'PDF 1.3', 'creationDate': '', 'page': 6, 'trapped': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'moddate': '2018-02-12T21:22:10-08:00', 'modDate': "D:20180212212210-08'00'", 'creationdate': '', 'creator': '', 'title': 'Attention is All you Need'}
Result 2: ID=doc_70528c47_106, Similarity Score=0.4636, Metadata={'moddate': '2018-02-12T21:22:10-08:00', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'trap

[{'id': 'doc_a461efd7_119',
  'document': 'convolution is equal to the combination of a self-attention layer and a point-wise feed-forward layer,\nthe approach we take in our model.\nAs side beneﬁt, self-attention could yield more interpretable models. We inspect attention distributions\nfrom our models and present and discuss examples in the appendix. Not only do individual attention\nheads clearly learn to perform different tasks, many appear to exhibit behavior related to the syntactic\nand semantic structure of the sentences.\n5\nTraining\nThis section describes the training regime for our models.\n5.1\nTraining Data and Batching\nWe trained on the standard WMT 2014 English-German dataset consisting of about 4.5 million\nsentence pairs. Sentences were encoded using byte-pair encoding [3], which has a shared source-\ntarget vocabulary of about 37000 tokens. For English-French, we used the signiﬁcantly larger WMT\n2014 English-French dataset consisting of 36M sentences and split toke

In [40]:
from langchain_groq import ChatGroq
import dotenv
import os
dotenv.load_dotenv()

groq_api_key = os.getenv("groq_api")

llm=ChatGroq(groq_api_key=groq_api_key,model="llama-3.1-8b-instant",temperature=0.3,max_tokens=1024)

def rag_response(query:str,retriever:Retriever, llm:ChatGroq,top_k:int=5):
    result=retriever.retrive(query,top_k=top_k)
    context = "\n\n".join([doc["document"] for doc in result]) if result else ""

    if not context:
        return "No relevant documents found."
    
    prompt = f"Use the following context to answer the question:\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"

    response = llm.invoke(prompt.format(context=context, query=query))
    return response.content
        

In [41]:
answer = rag_response("What is attention mechanism?",retriever,llm)
answer

Retrieving documents for query: 'What is attention mechanism?' with top_k=5 and score_threshold=0.0
Genrating embedding for text of length: 1


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.30it/s]


Genrated embedding with shape (1, 384)
Retrieved 5 results from the vector store.
Result 1: ID=doc_52222def_108, Similarity Score=0.4854, Metadata={'format': 'PDF 1.3', 'creator': '', 'moddate': '2018-02-12T21:22:10-08:00', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'title': 'Attention is All you Need', 'trapped': '', 'file_type': 'pdf', 'creationdate': '', 'source': 'Attention.pdf', 'content_length': 835, 'doc_index': 108, 'file_path': '../data/pdf/Attention.pdf', 'modDate': "D:20180212212210-08'00'", 'keywords': '', 'creationDate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'producer': 'PyPDF2', 'page': 3, 'total_pages': 11}
Result 2: ID=doc_a461efd7_119, Similarity Score=0.4814, Metadata={'title': 'Attention is All you Need', 'creationdate': '', 'moddate': '2018-02-12T21:22:10-08:00', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jone

"An attention mechanism is a technique used in deep learning models to allow the model to focus on specific parts of the input data when generating the output. It's a way for the model to selectively weigh the importance of different input elements when computing the output.\n\nIn the context of the Transformer model, the attention mechanism is used to allow the model to jointly attend to information from different representation subspaces at different positions. This is achieved by computing the dot product of the query and key vectors, which results in a weighted sum of the values.\n\nIn simpler terms, attention mechanism helps the model to:\n\n1. Focus on specific parts of the input data\n2. Weigh the importance of different input elements\n3. Compute the output based on the weighted sum of the input elements\n\nThere are different types of attention mechanisms, including:\n\n1. Self-attention: This type of attention mechanism is used to compute the representation of a single sequen